# Training Cost Benchmark Final64

Notebook ini digunakan untuk mengukur biaya komputasi training secara konsisten pada:

- Fixed Graph WaveNet
- Adaptive Graph WaveNet
- Hybrid Graph WaveNet

Setup benchmark:
- dataset: Final64;
- seed: 42;
- batch size: 32;
- maximum epoch: 300;
- early stopping;
- metrik waktu:
  - training time per epoch;
  - total training time;
  - best epoch.

Hasil benchmark kemudian dibandingkan dengan GC-LSTM.

In [1]:
from pathlib import Path
import sys
import time
import random

import numpy as np
import pandas as pd
import tensorflow as tf

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

CURRENT_DIR = Path.cwd().resolve()


def find_project_root(start_path: Path):
    candidates = [start_path] + list(start_path.parents)

    for path in candidates:
        if (path / "data" / "final_64").exists() and (path / "outputs" / "final_64").exists():
            return path

    raise FileNotFoundError("Project root tidak ditemukan.")


ROOT = find_project_root(CURRENT_DIR)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Project root:", ROOT)
print("TensorFlow:", tf.__version__)
print("Seed:", SEED)

Project root: /Users/aisha/Documents/SKRIPSI 2026/SEMINAR 2026/tomtom-stgnn-agent
TensorFlow: 2.21.0
Seed: 42


## Inspect Existing Graph WaveNet Training Setup

Sebelum mengukur training cost, konfigurasi training Graph WaveNet yang sudah digunakan sebelumnya diperiksa terlebih dahulu.

Tujuannya agar benchmark menggunakan:
- arsitektur yang sama;
- optimizer yang sama;
- learning rate yang sama;
- batch size yang sama;
- early stopping yang sama;
- maximum epoch yang sama.

In [2]:
from pathlib import Path

NOTEBOOK_DIR = ROOT / "notebooks"

print("Notebook files containing GraphWaveNet:\n")

for p in sorted(NOTEBOOK_DIR.glob("*.ipynb")):
    name_lower = p.name.lower()

    if "graphwavenet" in name_lower or "graph_wave" in name_lower or "gwn" in name_lower:
        print("-", p.name)

Notebook files containing GraphWaveNet:



## Locate Existing Graph WaveNet Notebook / Script

Pencarian dilakukan secara recursive dari project root untuk menemukan file notebook atau script yang berkaitan dengan Graph WaveNet.

In [3]:
# Cari file Graph WaveNet secara recursive

patterns = ["*graphwavenet*", "*graph_wave*", "*gwn*"]

found = []

for pattern in patterns:
    found.extend(ROOT.rglob(pattern))

# Hilangkan duplikat
found = sorted(set(found))

print("Graph WaveNet related files:\n")

for p in found:
    if p.is_file():
        print("-", p.relative_to(ROOT))

Graph WaveNet related files:

- notebooks/notebooks/graphwavenet_final64_audited.ipynb
- notebooks/notebooks/graphwavenet_final_analysis_demo.ipynb
- outputs/final_64/graphwavenet_audited/graphwavenet_adaptive_seed42.weights.h5
- outputs/final_64/graphwavenet_audited/graphwavenet_adaptive_seed42_adjacency.npy
- outputs/final_64/graphwavenet_audited/graphwavenet_adaptive_seed42_history.csv
- outputs/final_64/graphwavenet_audited/graphwavenet_adaptive_seed42_results.csv
- outputs/final_64/graphwavenet_audited/graphwavenet_audited_seed42_comparison.csv
- outputs/final_64/graphwavenet_audited/graphwavenet_fixed_seed42.weights.h5
- outputs/final_64/graphwavenet_audited/graphwavenet_fixed_seed42_history.csv
- outputs/final_64/graphwavenet_audited/graphwavenet_fixed_seed42_results.csv
- outputs/final_64/graphwavenet_audited/graphwavenet_fixed_vs_adaptive_seed42.csv
- outputs/final_64/graphwavenet_audited/graphwavenet_hybrid_seed42.weights.h5
- outputs/final_64/graphwavenet_audited/graphwavene

## Inspect Original Graph WaveNet Training Configuration

Notebook `graphwavenet_final64_audited.ipynb` digunakan sebagai sumber konfigurasi training asli.

Bagian yang diperiksa:
- optimizer dan learning rate;
- batch size;
- maximum epoch;
- early stopping;
- learning-rate scheduler;
- loss function;
- proses `model.fit()`.

Benchmark training akan mengikuti konfigurasi ini agar hasil computation cost tetap konsisten dengan eksperimen sebelumnya.

In [ ]:
import json

GWN_NOTEBOOK_PATH = ROOT / "notebooks" / "graphwavenet_final64_audited.ipynb"

with open(GWN_NOTEBOOK_PATH, "r", encoding="utf-8") as f:
    nb = json.load(f)

keywords = [
    "compile(",
    "fit(",
    "EarlyStopping",
    "ReduceLROnPlateau",
    "MAX_EPOCHS",
    "BATCH_SIZE",
    "learning_rate",
    "Adam(",
]

print("=== RELEVANT TRAINING CELLS ===\n")

for i, cell in enumerate(nb["cells"]):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    if any(keyword in source for keyword in keywords):
        print(f"\n--- CELL {i} ---\n")
        print(source)

## Benchmark Training Time — Fixed Graph WaveNet

Fixed Graph WaveNet dilatih ulang menggunakan konfigurasi asli:

- seed: 42
- Adam, learning rate 1e-3
- batch size: 32
- maximum epoch: 300
- masked MSE loss
- EarlyStopping patience: 25
- ReduceLROnPlateau patience: 8

Waktu yang dicatat:
- waktu tiap epoch;
- rata-rata waktu per epoch;
- total training time;
- best epoch.

In [ ]:
# ============================================================
# LOAD ORIGINAL GWN DEFINITIONS FROM NOTEBOOK
# ============================================================

import json

GWN_NOTEBOOK_PATH = ROOT / "notebooks" / "graphwavenet_final64_audited.ipynb"

with open(GWN_NOTEBOOK_PATH, "r", encoding="utf-8") as f:
    gwn_nb = json.load(f)

# Ambil semua code cell sampai sebelum training Fixed.
# Ini supaya class model, loss, data setup, graph support, dll sama.
code_cells = []

for idx, cell in enumerate(gwn_nb["cells"]):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    # stop sebelum cell training fixed asli
    if "fixed_history = fixed_model.fit(" in source:
        break

    code_cells.append(source)

print("Code cells collected:", len(code_cells))

## Initialize Original Graph WaveNet Environment

Definisi model, data, graph support, konfigurasi, dan loss dari notebook Graph WaveNet asli dimuat kembali agar benchmark training menggunakan setup yang sama.

In [ ]:
# ============================================================
# EXECUTE ORIGINAL SETUP CELLS
# ============================================================

for i, source in enumerate(code_cells):
    try:
        exec(source, globals())
    except Exception as e:
        print(f"Error pada extracted cell {i}:")
        print(type(e).__name__, e)
        raise RuntimeError(f"Failed to load GWN setup cell {i}") from e

print("\nOriginal Graph WaveNet setup loaded.")

## Fixed Graph WaveNet Training Benchmark

Fixed Graph WaveNet dilatih ulang menggunakan konfigurasi asli dan ditambahkan pencatatan waktu untuk mengukur:

- waktu setiap epoch;
- rata-rata waktu per epoch;
- total training time;
- best epoch;
- best validation loss.

In [8]:
class EpochTimer(tf.keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        self.epoch_times = []

    def on_epoch_begin(self, epoch, logs=None):
        self.start_time = time.perf_counter()

    def on_epoch_end(self, epoch, logs=None):
        elapsed = time.perf_counter() - self.start_time
        self.epoch_times.append(elapsed)


fixed_timer = EpochTimer()

fixed_callbacks_benchmark = [
    fixed_timer,
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=25, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=8, min_lr=1e-5, verbose=1
    ),
]

print("Fixed benchmark callbacks ready.")

Fixed benchmark callbacks ready.


In [9]:
# ============================================================
# RESET + REBUILD FIXED MODEL
# ============================================================

tf.keras.backend.clear_session()

np.random.seed(SEED)
tf.random.set_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

fixed_benchmark_model = GraphWaveNetModel(
    mode="fixed",
    fixed_supports=FIXED_SUPPORTS,
    num_nodes=NUM_NODES,
    num_horizons=NUM_HORIZONS,
    name="graph_wavenet_fixed_benchmark",
)

_ = fixed_benchmark_model(tf.convert_to_tensor(X_train[:1], dtype=tf.float32), training=False)

fixed_benchmark_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=GW_LEARNING_RATE), loss=masked_mse
)

print("Parameters:", fixed_benchmark_model.count_params())

# ============================================================
# TRAIN WITH TIMER
# ============================================================

fixed_train_start = time.perf_counter()

fixed_benchmark_history = fixed_benchmark_model.fit(
    X_train,
    y_train_masked,
    validation_data=(X_val, y_val_masked),
    epochs=GW_MAX_EPOCHS,
    batch_size=GW_BATCH_SIZE,
    callbacks=fixed_callbacks_benchmark,
    verbose=1,
)

fixed_total_training_time = time.perf_counter() - fixed_train_start

fixed_epoch_times = np.array(fixed_timer.epoch_times)

fixed_history_df = pd.DataFrame(fixed_benchmark_history.history)

fixed_best_epoch = fixed_history_df["val_loss"].idxmin() + 1

fixed_best_val_loss = fixed_history_df["val_loss"].min()

print("\n=== FIXED GWN TRAINING BENCHMARK ===")
print("Epochs run       :", len(fixed_epoch_times))
print("Best epoch       :", fixed_best_epoch)
print("Mean sec/epoch   :", round(fixed_epoch_times.mean(), 4))
print("Total train time :", round(fixed_total_training_time, 4), "seconds")
print("Best val loss    :", round(fixed_best_val_loss, 6))

Parameters: 276291
Epoch 1/300


E0000 00:00:1790151703.858025 1774413 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
/Users/aisha/Documents/SKRIPSI 2026/SEMINAR 2026/tomtom-stgnn-agent/.venv/lib/python3.11/site-packages/keras/src/optimizers/base_optimizer.py:862: UserWarning: Gradients do not exist for variables ['graph_wavenet_fixed_benchmark/graph_wavenet_backbone/gw_block_4_layer_2/diffusion_graph_conv_7/dense_7/kernel', '

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 197ms/step - loss: 0.3840

E0000 00:00:1790151715.155904 1774413 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


13/13 ━━━━━━━━━━━━━━━━━━━━ 12s 274ms/step - loss: 0.3840 - val_loss: 0.4851 - learning_rate: 0.0010
Epoch 2/300
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 238ms/step - loss: 0.2517 - val_loss: 0.4846 - learning_rate: 0.0010
Epoch 3/300
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 241ms/step - loss: 0.2305 - val_loss: 0.4585 - learning_rate: 0.0010
Epoch 4/300
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 236ms/step - loss: 0.2193 - val_loss: 0.4437 - learning_rate: 0.0010
Epoch 5/300
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 243ms/step - loss: 0.2097 - val_loss: 0.4350 - learning_rate: 0.0010
Epoch 6/300
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 248ms/step - loss: 0.2006 - val_loss: 0.4210 - learning_rate: 0.0010
Epoch 7/300
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 238ms/step - loss: 0.1934 - val_loss: 0.4029 - learning_rate: 0.0010
Epoch 8/300
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 246ms/step - loss: 0.1863 - val_loss: 0.3912 - learning_rate: 0.0010
Epoch 9/300
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 236ms/step - loss: 0.1813 - val_loss: 0.3576 - learning_rate: 0.0010
Epoch 10/300

## Adaptive Graph WaveNet Training Benchmark

Adaptive Graph WaveNet dilatih ulang menggunakan konfigurasi training yang sama dengan Fixed Graph WaveNet.

Waktu yang dicatat:
- waktu tiap epoch;
- rata-rata waktu per epoch;
- total training time;
- best epoch;
- best validation loss.

In [10]:
adaptive_timer = EpochTimer()

adaptive_callbacks_benchmark = [
    adaptive_timer,
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=25, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=8, min_lr=1e-5, verbose=1
    ),
]

# ============================================================
# RESET + REBUILD ADAPTIVE MODEL
# ============================================================

tf.keras.backend.clear_session()

np.random.seed(SEED)
tf.random.set_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

adaptive_benchmark_model = GraphWaveNetModel(
    mode="adaptive",
    fixed_supports=FIXED_SUPPORTS,
    num_nodes=NUM_NODES,
    num_horizons=NUM_HORIZONS,
    name="graph_wavenet_adaptive_benchmark",
)

_ = adaptive_benchmark_model(tf.convert_to_tensor(X_train[:1], dtype=tf.float32), training=False)

adaptive_benchmark_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=GW_LEARNING_RATE), loss=masked_mse
)

print("Parameters:", adaptive_benchmark_model.count_params())

# ============================================================
# TRAIN
# ============================================================

adaptive_train_start = time.perf_counter()

adaptive_benchmark_history = adaptive_benchmark_model.fit(
    X_train,
    y_train_masked,
    validation_data=(X_val, y_val_masked),
    epochs=GW_MAX_EPOCHS,
    batch_size=GW_BATCH_SIZE,
    callbacks=adaptive_callbacks_benchmark,
    verbose=1,
)

adaptive_total_training_time = time.perf_counter() - adaptive_train_start

adaptive_epoch_times = np.array(adaptive_timer.epoch_times)

adaptive_history_df = pd.DataFrame(adaptive_benchmark_history.history)

adaptive_best_epoch = adaptive_history_df["val_loss"].idxmin() + 1

adaptive_best_val_loss = adaptive_history_df["val_loss"].min()

print("\n=== ADAPTIVE GWN TRAINING BENCHMARK ===")
print("Epochs run       :", len(adaptive_epoch_times))
print("Best epoch       :", adaptive_best_epoch)
print("Mean sec/epoch   :", round(adaptive_epoch_times.mean(), 4))
print("Total train time :", round(adaptive_total_training_time, 4), "seconds")
print("Best val loss    :", round(adaptive_best_val_loss, 6))

Parameters: 261187
Epoch 1/300


E0000 00:00:1790152030.240336 1774413 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
/Users/aisha/Documents/SKRIPSI 2026/SEMINAR 2026/tomtom-stgnn-agent/.venv/lib/python3.11/site-packages/keras/src/optimizers/base_optimizer.py:862: UserWarning: Gradients do not exist for variables ['graph_wavenet_adaptive_benchmark/graph_wavenet_backbone/gw_block_4_layer_2/diffusion_graph_conv_7/dense_7/kernel'

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 206ms/step - loss: 0.4009

E0000 00:00:1790152041.815408 1774413 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


13/13 ━━━━━━━━━━━━━━━━━━━━ 12s 280ms/step - loss: 0.4009 - val_loss: 0.5528 - learning_rate: 0.0010
Epoch 2/300
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 208ms/step - loss: 0.2462 - val_loss: 0.4967 - learning_rate: 0.0010
Epoch 3/300
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 217ms/step - loss: 0.2303 - val_loss: 0.4936 - learning_rate: 0.0010
Epoch 4/300
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 211ms/step - loss: 0.2191 - val_loss: 0.4942 - learning_rate: 0.0010
Epoch 5/300
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 211ms/step - loss: 0.2135 - val_loss: 0.4855 - learning_rate: 0.0010
Epoch 6/300
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 216ms/step - loss: 0.2078 - val_loss: 0.4643 - learning_rate: 0.0010
Epoch 7/300
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 213ms/step - loss: 0.2005 - val_loss: 0.4525 - learning_rate: 0.0010
Epoch 8/300
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 214ms/step - loss: 0.1939 - val_loss: 0.4518 - learning_rate: 0.0010
Epoch 9/300
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 245ms/step - loss: 0.1884 - val_loss: 0.4395 - learning_rate: 0.0010
Epoch 10/300

## Hybrid Graph WaveNet Training Benchmark

Hybrid Graph WaveNet dilatih ulang dengan konfigurasi training yang sama.

Hybrid menggunakan tiga graph support:
- forward geographic;
- reverse geographic;
- adaptive learned adjacency.

Waktu training dicatat untuk dibandingkan dengan Fixed, Adaptive, dan GC-LSTM.

In [11]:
hybrid_timer = EpochTimer()

hybrid_callbacks_benchmark = [
    hybrid_timer,
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=25, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=8, min_lr=1e-5, verbose=1
    ),
]

# ============================================================
# RESET + REBUILD HYBRID MODEL
# ============================================================

tf.keras.backend.clear_session()

np.random.seed(SEED)
tf.random.set_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

hybrid_benchmark_model = GraphWaveNetModel(
    mode="hybrid",
    fixed_supports=FIXED_SUPPORTS,
    num_nodes=NUM_NODES,
    num_horizons=NUM_HORIZONS,
    name="graph_wavenet_hybrid_benchmark",
)

_ = hybrid_benchmark_model(tf.convert_to_tensor(X_train[:1], dtype=tf.float32), training=False)

hybrid_benchmark_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=GW_LEARNING_RATE), loss=masked_mse
)

print("Parameters:", hybrid_benchmark_model.count_params())

# ============================================================
# TRAIN
# ============================================================

hybrid_train_start = time.perf_counter()

hybrid_benchmark_history = hybrid_benchmark_model.fit(
    X_train,
    y_train_masked,
    validation_data=(X_val, y_val_masked),
    epochs=GW_MAX_EPOCHS,
    batch_size=GW_BATCH_SIZE,
    callbacks=hybrid_callbacks_benchmark,
    verbose=1,
)

hybrid_total_training_time = time.perf_counter() - hybrid_train_start

hybrid_epoch_times = np.array(hybrid_timer.epoch_times)

hybrid_history_df = pd.DataFrame(hybrid_benchmark_history.history)

hybrid_best_epoch = hybrid_history_df["val_loss"].idxmin() + 1

hybrid_best_val_loss = hybrid_history_df["val_loss"].min()

print("\n=== HYBRID GWN TRAINING BENCHMARK ===")
print("Epochs run       :", len(hybrid_epoch_times))
print("Best epoch       :", hybrid_best_epoch)
print("Mean sec/epoch   :", round(hybrid_epoch_times.mean(), 4))
print("Total train time :", round(hybrid_total_training_time, 4), "seconds")
print("Best val loss    :", round(hybrid_best_val_loss, 6))

Parameters: 293955
Epoch 1/300


E0000 00:00:1790152348.910336 1774413 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
/Users/aisha/Documents/SKRIPSI 2026/SEMINAR 2026/tomtom-stgnn-agent/.venv/lib/python3.11/site-packages/keras/src/optimizers/base_optimizer.py:862: UserWarning: Gradients do not exist for variables ['graph_wavenet_hybrid_benchmark/graph_wavenet_backbone/gw_block_4_layer_2/diffusion_graph_conv_7/dense_7/kernel', 

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 282ms/step - loss: 0.3780

E0000 00:00:1790152361.550097 1774413 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


13/13 ━━━━━━━━━━━━━━━━━━━━ 14s 363ms/step - loss: 0.3780 - val_loss: 0.4492 - learning_rate: 0.0010
Epoch 2/300
13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 296ms/step - loss: 0.2349 - val_loss: 0.4511 - learning_rate: 0.0010
Epoch 3/300
13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 288ms/step - loss: 0.2153 - val_loss: 0.4476 - learning_rate: 0.0010
Epoch 4/300
13/13 ━━━━━━━━━━━━━━━━━━━━ 5s 348ms/step - loss: 0.2023 - val_loss: 0.4426 - learning_rate: 0.0010
Epoch 5/300
13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 341ms/step - loss: 0.1945 - val_loss: 0.4391 - learning_rate: 0.0010
Epoch 6/300
13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 325ms/step - loss: 0.1847 - val_loss: 0.4690 - learning_rate: 0.0010
Epoch 7/300
13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 324ms/step - loss: 0.1796 - val_loss: 0.4529 - learning_rate: 0.0010
Epoch 8/300
13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 341ms/step - loss: 0.1747 - val_loss: 0.4482 - learning_rate: 0.0010
Epoch 9/300
13/13 ━━━━━━━━━━━━━━━━━━━━ 5s 350ms/step - loss: 0.1693 - val_loss: 0.4189 - learning_rate: 0.0010
Epoch 10/300

In [13]:
# ============================================================
# SAVE TRAINING COST BENCHMARK
# ============================================================

TRAINING_COST_OUTPUT = ROOT / "outputs" / "final_64" / "training_cost_benchmark_seed42.csv"

training_cost = pd.DataFrame(
    [
        {
            "model": "Fixed GWN",
            "mean_sec_per_epoch": 3.2293,
            "total_training_time_sec": 284.6711,
            "best_epoch_benchmark": 63,
        },
        {
            "model": "Adaptive GWN",
            "mean_sec_per_epoch": 2.9060,
            "total_training_time_sec": 267.8679,
            "best_epoch_benchmark": 67,
        },
        {
            "model": "Hybrid GWN",
            "mean_sec_per_epoch": 3.8203,
            "total_training_time_sec": 401.6228,
            "best_epoch_benchmark": 80,
        },
        {
            "model": "GC-LSTM",
            "mean_sec_per_epoch": 0.2540,
            "total_training_time_sec": 23.0799,
            "best_epoch_benchmark": 65,
        },
    ]
)

training_cost.to_csv(TRAINING_COST_OUTPUT, index=False)

display(training_cost)

print("\nSaved:")
print(TRAINING_COST_OUTPUT)

,model,mean_sec_per_epoch,total_training_time_sec,best_epoch_benchmark
0,Fixed GWN,3.2293,284.6711,63
1,Adaptive GWN,2.9060,267.8679,67
2,Hybrid GWN,3.8203,401.6228,80
3,GC-LSTM,0.2540,23.0799,65



Saved:
/Users/aisha/Documents/SKRIPSI 2026/SEMINAR 2026/tomtom-stgnn-agent/outputs/final_64/training_cost_benchmark_seed42.csv
